# N128 · 模运算、快速幂与逆元

**目标：** 在合法代数条件下化简大数计算。

**先修：** N126, N124, N063。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 模同余；快速幂；互素逆元；Fermat前提；CRT扩展；复合模数。

**配套讲解来源：** [同名逐章意见](../../comment/128_modular_arithmetic_fast_power.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章要解决的问题是：**乘方运算的数字会大到没有意义（比如 3 的 13 次方还好，3 的 10 万次方有将近五万位），但题目往往只要求"除以某个模数之后的余数"。怎么在数字始终不失控的前提下，把 a 的 n 次方对 mod 取余算出来？** 顺带的第二个问题：模运算里"除法"不像加减乘那样可以直接做，什么时候能除、怎么除？

给你具体到数字的例子：

- 输入：`a = 3, n = 13, mod = 17`
- 输出：`12`
- 为什么是它？因为 3¹³ = 1594323，而 1594323 = 17 × 93783 + 12，余数是 12。注意快速幂全程只跟 17 以内的数打交道，从没真正构造出 1594323 这个大数。

再看逆元的例子：输入 `a = 3, mod = 11`，`mod_inverse` 输出 `4`，因为 3 × 4 = 12，12 除以 11 余 1——也就是说在"模 11 的世界"里，4 扮演着 3 的倒数角色，乘 4 等价于除以 3。

## 2. 基础知识：先读懂这些词

- **模运算（同余）**：只关心"除以 mod 之后剩多少"的算术。两个数除以 mod 余数相同就叫同余，记作 a ≡ b (mod m)。比如 12 ≡ 1 (mod 11)。
- **加减乘可以直接化简**：(a+b) mod m、(a·b) mod m 可以先分别取余再算、算完再取余，结果不变。这是快速幂能一直保持小数字的法律依据。
- **除法不行、逆元来补**：模世界里没有真正的除法。想"除以 a"，只能改乘一个特殊数 a⁻¹，使 a·a⁻¹ ≡ 1 (mod m)。这个 a⁻¹ 叫 a 的逆元。
- **互素**：两个数的最大公约数为 1。逆元存在的条件是 a 与模数互素——比如 mod = 11 时 3 与 11 互素，逆元存在；而 mod = 12 时 gcd(6,12)=6，6 就没有逆元。
- **快速幂（binary exponentiation）**：把指数 n 看成二进制，逐位处理：底数不断自己平方（a → a² → a⁴ → a⁸…），指数每轮除以 2；当前二进制位是 1，就把当前底数乘进结果。n 个 a 相乘从"乘 n 次"降到"乘约 log₂n 次"。
- **扩展欧几里得算法**：普通欧几里得算法求最大公约数，扩展版顺手求出系数 x、y，满足 a·x + m·y = gcd(a,m)。当 gcd = 1 时，a·x ≡ 1 (mod m)，那个 x 就是逆元。这是求逆元的通用方法，模数不是质数也能用。
- **费马小定理的前提**：如果模数 p 是质数，可以用 a^(p−2) mod p 直接算逆元（配合快速幂）。但这个做法**只对质数模数成立**，模数是合数时会算错——notebook “正确性与复杂度”部分 特别强调"不盲用费马小定理处理复合模数"，所以本章实现走更通用的扩展 GCD。
- **超长指数（super pow）**：当指数本身是超长的十进制数（比如几千位），不用把它转成完整整数，可以按十进制逐位读入，套恒等式 a^(10e+d) = (a^e)¹⁰ · a^d。
- **复合模数（合数模数）**：模数不是质数的情形（比如本章的 1337 = 7 × 191）。费马小定理这时失效，但本章两条主力路线——快速幂、扩展 GCD 求逆——都不挑模数：快速幂只用乘法同余性质，扩展 GCD 只要互素就行。
- **CRT 扩展（中国剩余定理，“学习目标”部分 列为知识点的延伸方向）**：把"对一个复合模数取余"拆成"对若干个两两互素的质数幂分别取余再合并"的定理。本章只把它列为知识地图上的下一站，代码实现没有用到；你只需要知道，遇到复合模数下的大计算时，CRT 是标准的拆解工具。

## 3. 思路推导：从小例子开始

- **Step 1：接受"只留余数"的设定。** 一切运算做完立刻 % mod，数字永远压在 mod 以内，加减乘都不受影响。
- **Step 2：把指数写成二进制。** 比如 13 的二进制是 1101，即 13 = 8 + 4 + 1，所以 3¹³ = 3⁸ · 3⁴ · 3¹。看出门道了吗——只需要准备 3¹、3²、3⁴、3⁸ 这一串"翻倍幂"，按二进制位挑几个乘起来。
- **Step 3：逐位扫描二进制。** 维护两个量：当前底数 a（不断平方）和累计结果 result。指数 n 每轮右移一位（`n >>= 1`），如果刚移出去的那位是 1（`n & 1`），就把当前底数乘进 result。
- **Step 4：除法问题交给扩展 GCD。** 求 a⁻¹ 就解方程 a·x ≡ 1 (mod m)，等价于 a·x + m·y = 1，用扩展欧几里得解出 x 再 % m 归一。
- **Step 5：超长指数逐位十进制处理。** 读入一位数字 d 时，旧答案 answer 代表 a^e，新指数变成 10e+d，于是新答案 = answer¹⁰ · a^d（都取模）。

**手算演示一：mod_pow(3, 13, 17) 全过程**（这正是 “运行示例”部分 自动生成的表格，最终打印"最终结果 12"）：

| 剩余指数 n | 当前底数 a | 累计结果 result | 该位是 1 吗（要不要乘） |
|-----------|-----------|----------------|------------------------|
| 13 | 3 | 1 | 1，乘：result = 1×3 % 17 = 3 |
| 6 | 9 | 3 | 0，不乘 |
| 3 | 13 | 3 | 1，乘：result = 3×13 % 17 = 5 |
| 1 | 16 | 5 | 1，乘：result = 5×16 % 17 = 12 |

逐行核对：初始 result=1、a=3。第一行后 a 平方成 9、n 折半成 6；第二行 n 是偶数跳过乘法，a 平方成 9²%17 = 13、n 变 3；第三行乘完 a 平方成 13²%17 = 16、n 变 1；第四行乘完 n 变 0，循环结束，返回 12。13 的二进制 1101 从低位到高位恰好是 1、0、1、1，和"要不要乘"那列完全对应。

**手算演示二：mod_inverse(3, 11)。** 扩展 GCD 每轮做带余除法并同步更新系数，这里只看关键量（不变的约定：old_r ≡ 3·old_x (mod 11)）：

| 轮次 | 除法 | 余数变化 | x 系数变化 |
|------|------|----------|-----------|
| 1 | 3 = 0×11 + 3 | (3,11) → (11,3) | (1,0) → (0,1) |
| 2 | 11 = 3×3 + 2 | (11,3) → (3,2) | (0,1) → (1,−3) |
| 3 | 3 = 1×2 + 1 | (3,2) → (2,1) | (1,−3) → (−3,4) |
| 4 | 2 = 2×1 + 0 | (2,1) → (1,0)，结束 | x = 4 |

余数到 1 说明 gcd = 1，逆元存在，x = 4 再 % 11 得 4。验算：3×4 = 12 ≡ 1 (mod 11)，正确。

**手算演示三：super_pow(2, [1, 2])，即 2¹² mod 1337。** 第一轮 digit=1：answer = 1¹⁰ × 2¹ = 2；第二轮 digit=2：answer = 2¹⁰ × 2² = 1024 × 4 = 4096，4096 % 1337 = 85。对照 2¹² = 4096，一致。

再体会一下快速幂到底省了多少：13 的二进制是 1101，共 4 位，所以 `mod_pow(3,13,17)` 只循环 4 轮、每轮两三次乘法；如果老老实实乘 13 次再取模，n = 10⁹ 时差距就是"30 轮 vs 10 亿次"。指数每右移一位等于除以 2，这正是"log n 轮"的直观来源。

## 4. 核心代码：mod_pow

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def mod_pow(a, n, mod):
    if n < 0 or mod <= 0:
        raise ValueError('nonnegative exponent and positive modulus required')
    result = 1 % mod
    a %= mod
    while n:
        if n & 1:
            result = result * a % mod
        a = a * a % mod
        n >>= 1
    return result
```

### 逐段读懂代码

“分段实现”部分 有三个函数。

**函数一：`mod_pow(a, n, mod)` —— 计算 a^n mod m。**

```python
def mod_pow(a, n, mod):
    if n < 0 or mod <= 0:
        raise ValueError('nonnegative exponent and positive modulus required')
    result = 1 % mod
    a %= mod
    while n:
        if n & 1:
            result = result * a % mod
        a = a * a % mod
        n >>= 1
    return result
```

- `if n<0 or mod<=0: raise ValueError(...)`：挡住两类非法输入——负指数（本章定义域外，负指数需要逆元，另走 `mod_inverse`）和非正模数（mod=0 会除零，mod 为负意义混乱）。
- `result=1%mod`：结果初始化为 1，但立刻 % mod。这步看似多余，其实专门伺候 mod=1：任何数 mod 1 都是 0，连"乘法单位元 1"也得变成 0（测试里 `mod_pow(4,0,1)==0` 测的就是它）。
- `a%=mod`：先把底数压进模内，允许调用方传大数或负数（Python 的 % 对负数也返回非负余数）。
- `while n:`：只要指数还没耗完就继续。n 每轮至少折半，循环次数约 log₂n。
- `if n&1: result=result*a%mod`：`n&1` 取 n 的最低二进制位。位是 1 说明当前底数该进入答案，乘完立刻取模。
- `a=a*a%mod; n>>=1`：底数平方（a → a² → a⁴…），指数右移一位。注意即使最低位是 0、这次没乘，平方照样要做，为下一轮备好"a 的下一个二次幂"。

**函数二：`mod_inverse(a, mod)` —— 扩展欧几里得求逆元。**

```python
def mod_inverse(a, mod):
    if mod < 2:
        raise ValueError('inverse modulus must be at least 2')
    old_r, r = (a % mod, mod)
    old_x, x = (1, 0)
    while r:
        q = old_r // r
        old_r, r = (r, old_r - q * r)
        old_x, x = (x, old_x - q * x)
    if old_r != 1:
        raise ValueError('inverse does not exist: gcd != 1')
    return old_x % mod
```

- `if mod<2: raise ValueError(...)`：挡住 mod<2。mod=1 的世界里所有数都是 0，谈逆元没有意义。
- `old_r,r=a%mod,mod; old_x,x=1,0`：两对变量。r 那对跑欧几里得余数序列，初值是 (a mod m, m)；x 那对同步记系数，初值 (1, 0) 对应"a×1 + m×0 = a mod m"这个起点。注意 Python 赋值 `old_r,r=r,old_r-q*r` 右边先整体求值再赋，天然保存了旧值，不需要临时变量。
- `while r:`：普通 GCD 循环，每轮 `q=old_r//r` 是商，`old_r-q*r` 是余数，余数序列严格递减直到 0。
- `old_x,x=x,old_x-q*x`：系数跟余数做完全相同的线性组合。数学上可以证明循环里始终成立 a·old_x + m·(某数) = old_r，这是整个算法的灵魂。
- `if old_r!=1: raise ValueError(...)`：循环结束时 old_r 是 gcd(a, mod)。gcd 不是 1 就没有逆元（比如 gcd(6,12)=6），直接报错拒绝，不返回垃圾值。
- `return old_x%mod`：解出的 x 可能是负数或超界，% mod 归一到 [0, mod)。

**函数三：`super_pow(a, digits)` —— 十进制逐位的超长指数幂，模数固定 1337。**

```python
def super_pow(a, digits):
    answer = 1
    for digit in digits:
        if not isinstance(digit, int) or not 0 <= digit <= 9:
            raise ValueError('decimal exponent digits required')
        answer = mod_pow(answer, 10, 1337) * mod_pow(a, digit, 1337) % 1337
    return answer
```

- `answer=1`：初始指数视为 0（a⁰ = 1），之后每读一位数字就"整体升十次方再乘 a^d"。
- `if not isinstance(digit,int) or not 0<=digit<=9: raise ValueError(...)`：挡住坏输入——元素不是整数（比如传了字符串 '3'）或者不是 0 到 9 的十进制数码（比如 42 混进列表）。
- `answer=mod_pow(answer,10,1337)*mod_pow(a,digit,1337)%1337`：一行实现恒等式 a^(10e+d) = (a^e)¹⁰ · a^d (mod 1337)。`mod_pow(answer,10,1337)` 把旧指数幂升十次方，`mod_pow(a,digit,1337)` 算新增的一位（digit ≤ 9，幂极小），乘起来再取模。全程没有把 digits 拼成大整数。
- 注意这题指数逐位更新的公式对任意正模数都成立，不要求模数是质数（“正确性与复杂度”部分 特意强调这点）。

## 5. 分段实现：按顺序运行

**本章接口：** `mod_pow(a, n, mod)`、`mod_inverse(a, mod)`、`super_pow(a, digits)`

### mod_pow

In [1]:
def mod_pow(a, n, mod):
    if n < 0 or mod <= 0:
        raise ValueError('nonnegative exponent and positive modulus required')
    result = 1 % mod
    a %= mod
    while n:
        if n & 1:
            result = result * a % mod
        a = a * a % mod
        n >>= 1
    return result

### mod_inverse

In [2]:
def mod_inverse(a, mod):
    if mod < 2:
        raise ValueError('inverse modulus must be at least 2')
    old_r, r = (a % mod, mod)
    old_x, x = (1, 0)
    while r:
        q = old_r // r
        old_r, r = (r, old_r - q * r)
        old_x, x = (x, old_x - q * x)
    if old_r != 1:
        raise ValueError('inverse does not exist: gcd != 1')
    return old_x % mod

### super_pow

In [3]:
def super_pow(a, digits):
    answer = 1
    for digit in digits:
        if not isinstance(digit, int) or not 0 <= digit <= 9:
            raise ValueError('decimal exponent digits required')
        answer = mod_pow(answer, 10, 1337) * mod_pow(a, digit, 1337) % 1337
    return answer

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [4]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a,n,mod=3,13,17; result=1; rows=[]
while n:
    rows.append((n,a,result,n&1))
    if n&1: result=result*a%mod
    a=a*a%mod; n>>=1
show_table(['剩余指数','当前底数','累计结果','是否相乘'],rows)
print('最终结果',result)

剩余指数,当前底数,累计结果,是否相乘
13,3,1,1
6,9,3,0
3,13,3,1
1,16,5,1


最终结果 12


## 7. 正确性、适用条件与复杂度

快速幂保持result·a^n与原目标同余，处理奇偶后不变量不变。扩展GCD给出ax+my=g，只有g=1时x才是逆元。指数逐位更新利用a^(10e+d)=(a^e)^10a^d，对任意正模数成立，不依赖模数为素数。

**代价：** 快速幂O(log n)次模乘；逆元O(log mod)级别除法；D位十进制指数O(D)次固定小指数模幂。均需计入大整数乘除位复杂度；不盲用费马小定理处理复合模数。

### 展开理解

**快速幂为什么正确？** 整个循环维持一条不变性质：result · a^n（用当前变量的值算）始终同余于最初想算的 a^n。每轮看指数奇偶：奇数就把一个 a 从 a^n 挪进 result（n 减 1 后变偶数），然后统一把 (a)² 和 n÷2 互换——a^(2k) = (a²)^k，这步交换不改变乘积。n 最终归零，a⁰ = 1，result 独自扛下了全部乘积，就是答案。

**逆元为什么正确？** 扩展 GCD 结束时满足 a·old_x + m·y = gcd(a,m)。两边同时 mod m，含 m 的项消失，剩下 a·old_x ≡ gcd (mod m)。gcd = 1 时 a·old_x ≡ 1，old_x 正是逆元；gcd ≠ 1 时可以证明逆元必然不存在（若 a·x ≡ 1 则 gcd 整除 1，矛盾），所以报错是有依据的，不是偷懒。

**super_pow 为什么正确？** 归纳着看：假设处理完前几位后 answer ≡ a^e。再来一位 d，新指数是 10e+d，而 a^(10e+d) = (a^e)¹⁰ · a^d，代入 answer 就得到新值。空列表时指数为 0，answer 保持 1，对应 a⁰ = 1。

**复杂度（结合具体数字）：** 快速幂 O(log n) 次模乘——n = 10⁹ 也只要约 30 轮（2³⁰ > 10⁹），每轮两三次乘法，总共百次以内的操作，瞬间完成。逆元 O(log mod) 次除法，mod = 10⁹ 同样是 30 轮上下。super_pow 是 D 位指数做 D 次小幂（每次固定 10 次方或 9 以内次方），D = 1000 的指数也只要一千来轮。唯一要留心的是 “正确性与复杂度”部分 提到的"大整数乘除的位复杂度"：如果 a 和 mod 本身是几百位的大数，一次乘法就不再是"一次操作"，要按位数另行估计。

## 8. 单元测试：每个用例在检查什么

1. `assert mod_inverse(3,11)==4 and mod_pow(4,0,1)==0` —— 前半是**正常值**：3 mod 11 的逆元是 4（3×4=12≡1）。后半是**特殊值**：mod=1 这个退化场景下，任何结果都该是 0，哪怕指数是 0、数学上 a⁰=1，取模后也必须是 0——专门检验 `result=1%mod` 那行初始化。
2. `for a in range(-12,13): for n in range(20): for mod in range(1,15): assert mod_pow(a,n,mod)==pow(a,n,mod)` —— 拿 Python 内建三参数 `pow` 当标准答案，穷举 25 个底数（含 0 和 12 个负数）、20 个指数、14 个模数（含 mod=1）共 7000 种组合逐一对照。负底数特别考察 `a%=mod` 归一后结果是否仍落在正确区间。
3. 逆元存在性双向检查：`gcd(a,mod)==1` 时验证 `a*mod_inverse(a,mod)%mod==1`（乘回去等于 1，这是逆元的定义）；`gcd≠1` 时用 try/except 要求**必须抛 ValueError**，如果居然返回了值就直接 `raise AssertionError('noninvertible value accepted')`——既测"该有的时候有"，也测"不该有的时候拒绝"。
4. `for n in range(80): assert super_pow(a,list(map(int,str(n))))==pow(a,n,1337)` —— 把 0 到 79 的数字转成十进制位列表喂给 super_pow，和内建 pow(·,·,1337) 对照。`list(map(int,str(n)))` 就是"把 42 变成 [4,2]"。
5. `assert super_pow(2,[])==1` —— **边界值**：空位列表代表指数 0，任何数的 0 次方是 1。

In [5]:
import math

assert mod_inverse(3, 11) == 4 and mod_pow(4, 0, 1) == 0

for a in range(-12, 13):
    for n in range(20):
        for mod in range(1, 15):
            assert mod_pow(a, n, mod) == pow(a, n, mod)
    for mod in range(2, 20):
        if math.gcd(a, mod) == 1:
            assert a * mod_inverse(a, mod) % mod == 1
        else:
            try:
                mod_inverse(a, mod)
            except ValueError:
                pass
            else:
                raise AssertionError('noninvertible value accepted')
    for n in range(80):
        assert super_pow(a, list(map(int, str(n)))) == pow(a, n, 1337)

assert super_pow(2, []) == 1

print('N128: 所有本章断言通过')

N128: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 比较扩展欧几里得求逆和素数模快速幂求逆。

**练习 2：** 构造不可逆元素。

<details>
<summary>展开思路提示（不是完整答案）</summary>

**练习 1（扩展 GCD 求逆 vs 质数模快速幂求逆）：** 提示——两条路最终给同一个答案，但适用范围不同：费马小定理 a^(p−2) 只在模数 p 是质数时成立，扩展 GCD 只要互素就行。设计实验：取质数模 11 和合数模 15 各试一组，看快速幂路线在合数模上是不是真的会出错（比如 mod=15、a=7 时对比两种方法结果）；再比较操作次数的感觉。手算示例可以先算 mod_inverse(3,11)=4，再用 3⁹ mod 11 验证费马路线也得 4。

**练习 2（构造不可逆元素）：** 提示——目标是找"和模数共享质因子的数"。抓手是 gcd：只要 gcd(a, mod) > 1，逆元就不存在。取模数 12（因子 2²×3），那么 2、3、4、6、8、9、10 全都不可逆；随便挑一个，手算说明原因（如果 a·x ≡ 1 (mod 12) 成立，gcd 就得整除 1，矛盾），然后调用 `mod_inverse` 看它是不是如预期抛出 ValueError。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [6]:
# 完整实现：本单元格不依赖其他单元格。
def mod_pow(a, n, mod):
    if n < 0 or mod <= 0:
        raise ValueError('nonnegative exponent and positive modulus required')
    result = 1 % mod
    a %= mod
    while n:
        if n & 1:
            result = result * a % mod
        a = a * a % mod
        n >>= 1
    return result

def mod_inverse(a, mod):
    if mod < 2:
        raise ValueError('inverse modulus must be at least 2')
    old_r, r = (a % mod, mod)
    old_x, x = (1, 0)
    while r:
        q = old_r // r
        old_r, r = (r, old_r - q * r)
        old_x, x = (x, old_x - q * x)
    if old_r != 1:
        raise ValueError('inverse does not exist: gcd != 1')
    return old_x % mod

def super_pow(a, digits):
    answer = 1
    for digit in digits:
        if not isinstance(digit, int) or not 0 <= digit <= 9:
            raise ValueError('decimal exponent digits required')
        answer = mod_pow(answer, 10, 1337) * mod_pow(a, digit, 1337) % 1337
    return answer

# 示例与回归测试
import math

assert mod_inverse(3, 11) == 4 and mod_pow(4, 0, 1) == 0

for a in range(-12, 13):
    for n in range(20):
        for mod in range(1, 15):
            assert mod_pow(a, n, mod) == pow(a, n, mod)
    for mod in range(2, 20):
        if math.gcd(a, mod) == 1:
            assert a * mod_inverse(a, mod) % mod == 1
        else:
            try:
                mod_inverse(a, mod)
            except ValueError:
                pass
            else:
                raise AssertionError('noninvertible value accepted')
    for n in range(80):
        assert super_pow(a, list(map(int, str(n)))) == pow(a, n, 1337)

assert super_pow(2, []) == 1

print('N128: 所有本章断言通过')

N128: 所有本章断言通过


## 11. 代表题与迁移

- **50. Pow(x, n)**：实数幂、无取模——练的是同一个"指数二进制分解、底数逐位平方"的骨架，只是要注意它没有 mod 来帮你压数字，且 n 可能为负。
- **372. Super Pow**：a 的超长十进制指数幂 mod 1337——就是本章 `super_pow` 的原型题，逐位读入 + a^(10e+d) 恒等式原样照用。
- **1922. Count Good Numbers**：数位组合计数取模 10⁹+7——练"组合数是偶数次幂与奇数次幂相乘"的建模，最终用 mod_pow 落地，考察本章快速幂直接当工具用。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。